In [18]:
from openai import OpenAI
from groq import Groq
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import Markdown, display
import os 
from builders.pdf_extractor import extract_text_pdf
from rich import print as rprint
import json

In [19]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

In [20]:
# models
gpt_oss = "openai/gpt-oss-120b"

In [21]:
def openai_llm_model(base_url, api_key, messages, model):
    client = OpenAI(base_url= base_url,  api_key=api_key)
    response = client.chat.completions.create(
    messages=messages,
    model=model,
    # stream= True,
    temperature=0
    )

    return response.choices[0].message.content

    

In [22]:
def groq_llm_model(messages: str, model: str, base_url: str = None, api_key : str = None):
    client = Groq(base_url= base_url,  api_key=api_key)
    response = client.chat.completions.create(
    messages=messages,
    model=model,
    # stream= True,
    temperature=0
    )

    return response.choices[0].message.content

In [23]:
resume_data = extract_text_pdf("./resume_ark/resume_original.pdf")

In [24]:
json_schema = {}
with open(r"./schema/resume_schema.json", "r") as schema:
    json_schema = json.load(schema)

In [25]:
job_description_ts = """ 
Omnicom Health is the largest healthcare marketing and communications network in the world—partnering with clients to achieve meaningful change through better, more innovative communications. With more than 5,000 dedicated healthcare communications specialists from every corner of science, medicine, marketing, and communications, we recognize that it takes the best of talent to deliver the best of health.

Title: 2026 Bold Beginners - Full Stack Developer Intern

Location: New York, NY

Department/Discipline: Technology/Engingeering

Responsibilities:

Learn full stack web development through actively engaging in training sessions, code reviews, and hands-on development projects
Support senior developers in designing, building, and maintaining web applications and internal tools
Develop and enhance user-facing features using modern front-end technologies (HTML, CSS, JavaScript, React, Vue, or Angular)
Build and optimize back-end APIs and database structures using server-side languages and frameworks
Collaborate with product managers, designers, and fellow developers to implement new features and functionality
Participate in debugging, testing, and troubleshooting of applications and code
Write clean, well-documented, and maintainable code following established coding standards and best practices
Assist in managing version control (Git), code repositories, and CI/CD pipelines
Attend and participate in weekly team meetings, standups, and code reviews
Contribute to technical documentation and knowledge sharing within the team
Experience:

Currently enrolled in or recently graduated from a Computer Science, Software Engineering, or related degree program
Strong foundational knowledge of web development concepts and architecture
Proficiency in at least one front-end language/framework (JavaScript, React, Vue, Angular, or similar)
Proficiency in at least one back-end language (Python, Node.js, Java, C#, PHP, or similar)
Experience with SQL and relational databases (MySQL, PostgreSQL, or similar)
Familiarity with Git and version control systems
Strong problem-solving skills and ability to learn new technologies quickly
Strong written and verbal communication, and critical thinking skills
Ability to work in a fast-paced environment, both independently and as part of a team
Excellent organizational skills, strong attention to detail, and ability to manage multiple tasks
Experience with RESTful APIs and understanding of HTTP protocols, a plus
Familiarity with cloud platforms (AWS, GCP, Azure) or containerization (Docker), a plus
Interest in healthcare technology or healthcare marketing applications, a plus
Portfolio or GitHub profile demonstrating previous projects and coding abilities, a plus
The range below represents the low and high end of the base salary someone in this role may earn as an employee of an Omnicom Health company in the United States.   Salaries will vary based on various factors including but not limited to professional and academic experience, training, associated responsibilities, and other business and organizational needs. The range listed is just one component of our total compensation package for employees. Salary decisions are dependent on the circumstances of each hire. 

$19 - $20/HR
 

Omnicom Health is committed to hiring and developing exceptional talent. We agree that talent is uniquely distributed, and we’re focused on developing inclusive teams that can bring the best solutions to everything we do. We strongly believe that celebrating what makes us different makes us better together. Join us—we look forward to getting to know you. We will process your personal data in accordance with our Recruitment Privacy Notice.
"""

In [26]:
system_prompt = f"""
You are an elite recruitment advisor and career coach with over 20 years of experience in talent acquisition, 
resume writing, and career development across diverse industries. You specialize in tailoring resumes to specific 
job descriptions, ensuring they are ATS-optimized and positioned to stand out in a competitive candidate pool.

The user will provide you with:
1. Their current resume text.
2. A target job description they want to apply for.

Your Task:
You are NOT an advisor giving suggestions — you are actively rewriting and tailoring the resume.
Your output must be a single, valid JSON object that strictly follows the schema provided below.
Do not include any text, explanation, or commentary outside of the JSON response.

How to tailor the resume:
- Rewrite the resume content so it is fully aligned with the provided job description.
- Naturally incorporate relevant keywords from the job posting to ensure ATS compatibility.
- Maintain a professional, achievement-focused tone throughout.
- Lead with a compelling Professional Summary that directly mirrors the language and priorities of the job description.
- Prioritize and highlight the candidate's most relevant skills, experiences, and accomplishments.
- Remove or minimize details that are unrelated to the target role.
- Use strong action verbs and quantify results wherever the existing data supports it.
- Reorder sections where necessary to front-load the most relevant qualifications.
- Suggest 2-3 bullet points the candidate could add to strengthen their application.

When processing the job description, you must:
1. Extract the job title, company name, required skills, qualifications, responsibilities, and emphasized keywords.
2. Identify the company's tone and culture from the language used in the posting.
3. Use that analysis to inform every section of the rewritten resume.

Content Rules:
- You may rephrase, reframe, or combine existing details to better match the role.
- If the job description cannot be read or is unclear, return a JSON response with an appropriate error message in the relevant field.
- Only process content related to career development and job applications.

Output Rules:
- Your response must be a single valid JSON object only — no markdown, no code fences, no explanation.
- Follow the exact JSON schema provided below. Do not change or rename any keys.
- You may only change the values to reflect the tailored resume content.

JSON Schema:
{str(json_schema)}
"""

In [27]:
user_prompt = f""" 
Here is my current resume:
<resume>
{resume_data}
</resume>

Here is the job description I want to apply for:
<job_description>
{job_description_ts}
</job_description>

Tailor my resume to this job description and return the result as a valid JSON object that strictly follows the schema provided.
"""


In [28]:
llm_message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [29]:
tailored_data = groq_llm_model(
    api_key= groq_key,
    model=gpt_oss,
    messages=llm_message
)

In [30]:
tailored_data = json.loads(tailored_data)

In [31]:
rprint(tailored_data)

{
    'success': True,
    'llm_notes': 'Suggested additional bullet points for the resume:\n1. Add a healthcare‑focused project or 
coursework (e.g., "Developed a HIPAA‑compliant patient portal prototype using React and FastAPI, improving data 
security and user experience.") to demonstrate direct relevance to Omnicom Health.\n2. Highlight participation in a
hackathon or team challenge that involved rapid prototyping of a web application, emphasizing fast‑paced, 
collaborative problem‑solving.\n3. Include a concise statement of interest in healthcare technology, such as 
"Passionate about leveraging full‑stack development to create innovative solutions for the healthcare marketing 
ecosystem."',
    'generated_data': {
        'company_name': 'Omnicom',
        'header': {
            'full_name': 'Richmond Bioh Nyantakyi',
            'email': 'richmondbhn03@gmail.com',
            'number': '(908)-567-9006',
            'location': 'Hillside, New Jersey',
            'linkdin': 'linkedin.com/in/richmond003',
            'github': 'github.com/richmond003'
        },
        'summary': 'Enthusiastic Computer Science junior with a Business Analytics minor, specializing in 
full‑stack web development and data‑driven solutions. Proven ability to build responsive React front‑ends, robust 
Python FastAPI back‑ends, and PostgreSQL databases, complemented by experience with Docker, CI/CD pipelines, and 
RESTful API integration. Strong communicator who thrives in fast‑paced, collaborative environments and is eager to 
apply technical expertise to healthcare technology at Omnicom Health.',
        'education': {
            'school': 'Caldwell University',
            'study': 'B.S. Computer Science, Minor in Business Analytics',
            'expected': 'May 2027',
            'GPA': 3.78,
            'award': 'Dean’s Award for Professional Excellence (2024/2025)',
            'Relevant Courses': [
                'Data Structures and Algorithms',
                'Full Stack Development',
                'Database Management Systems',
                'Business/Data Analytics',
                'Software Engineering',
                'Web Development'
            ]
        },
        'experience': [
            {
                'job_title': 'IT Support Assistant',
                'timeline': 'Feb 2024 – Present',
                'organization': 'Caldwell University — Board of Online Education',
                'location': 'Caldwell, NJ',
                'responsiblities': [
                    'Collaborated with faculty to deploy and maintain LMS platforms, reducing technical support 
tickets by 25% and improving user satisfaction.',
                    'Configured, imaged, and deployed over 75 Windows and macOS workstations each semester, cutting
setup time by 40% through automated scripting.',
                    'Resolved first‑level ServiceNow tickets with a 95% satisfaction rating, demonstrating strong 
troubleshooting and communication skills.'
                ]
            },
            {
                'job_title': 'IT Extern',
                'timeline': 'Aug 2025 – Dec 2025',
                'organization': 'YMCA of Livingston',
                'location': 'Livingston, NJ',
                'responsiblities': [
                    'Executed software installations, security patches, and automated backup routines across 50+ 
workstations, ensuring compliance with cybersecurity standards.',
                    'Diagnosed and repaired network and printer issues, boosting system uptime by 15% for staff and
volunteers.',
                    'Documented procedures in Confluence and used Git for version control of configuration scripts,
supporting team knowledge sharing.'
                ]
            },
            {
                'job_title': 'Research Assistant — AI‑Powered Student Performance Prediction',
                'timeline': 'Sep 2024 – Dec 2024',
                'organization': 'Caldwell University',
                '

In [32]:
from builders.docx_builder import Resume_Builder

if tailored_data["success"]:
    doc = Resume_Builder(tailored_data["generated_data"])
else:
    print("failed")

  0%|          | 0/1 [00:00<?, ?it/s]

Saved: resume_ark\Omnicom_resume.pdf


In [33]:
display(Markdown(tailored_data["llm_notes"]))

Suggested additional bullet points for the resume:
1. Add a healthcare‑focused project or coursework (e.g., "Developed a HIPAA‑compliant patient portal prototype using React and FastAPI, improving data security and user experience.") to demonstrate direct relevance to Omnicom Health.
2. Highlight participation in a hackathon or team challenge that involved rapid prototyping of a web application, emphasizing fast‑paced, collaborative problem‑solving.
3. Include a concise statement of interest in healthcare technology, such as "Passionate about leveraging full‑stack development to create innovative solutions for the healthcare marketing ecosystem."

In [34]:
rprint(tailored_data["llm_notes"])

Suggested additional bullet points for the resume:
1. Add a healthcare‑focused project or coursework (e.g., "Developed a HIPAA‑compliant patient portal prototype 
using React and FastAPI, improving data security and user experience.") to demonstrate direct relevance to Omnicom 
Health.
2. Highlight participation in a hackathon or team challenge that involved rapid prototyping of a web application, 
emphasizing fast‑paced, collaborative problem‑solving.
3. Include a concise statement of interest in healthcare technology, such as "Passionate about leveraging 
full‑stack development to create innovative solutions for the healthcare marketing ecosystem."